# Download Videos From Dryad

Use this notebook to fetch raw videos into `video/` before running per-date analysis notebooks.

Workflow:
1. Run Cell 2 after the Dryad dataset is published.
2. It discovers the published files from the dataset DOI and downloads the four GoPro videos into `video/`.
3. Set `DOWNLOAD_ARDUCAM_ZIP = True` to also download the single zipped Arducam recording.
4. The downloader verifies SHA-256 checksums whenever Dryad supplies them.

In [1]:
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root from the current notebook working directory."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'data').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root (expected data/ and scripts/).')

ROOT = find_repo_root()
VIDEO_DIR = ROOT / 'video'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

DRYAD_DOI = '10.5061/dryad.x0k6djj17'
DOWNLOAD_ARDUCAM_ZIP = False
EXPECTED_GOPRO_FILES = [
    'GX010063_2025Nov10.MP4',
    'GX010067_2025Nov12.MP4',
    'GX010072_2025Nov13.MP4',
    'GX01007X_2025Nov14.MP4',
]
DOWNLOADER = ROOT / 'scripts' / 'download_dryad_videos.py'

print('Dryad DOI:', DRYAD_DOI)
print('Target directory:', VIDEO_DIR)
print('Downloader:', DOWNLOADER)
print('Download Arducam ZIP:', DOWNLOAD_ARDUCAM_ZIP)

Dryad DOI: 10.5061/dryad.x0k6djj17
Target directory: /Users/oakley/Documents/GitHub/signal_respirometry/video
Files listed: 4


In [2]:
import subprocess
import sys

command = [sys.executable, str(DOWNLOADER), '--doi', DRYAD_DOI, '--output-dir', str(VIDEO_DIR)]
if DOWNLOAD_ARDUCAM_ZIP:
    command.append('--include-arducam-zip')

print('Running:', ' '.join(command))
subprocess.run(command, check=True)

Skipping existing file: GX010063_2025Nov10.MP4
Skipping existing file: GX010067_2025Nov12.MP4
Skipping existing file: GX010072_2025Nov13.MP4
Skipping existing file: GX01007X_2025Nov14.MP4


In [3]:
import hashlib

def sha256sum(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

for name in EXPECTED_GOPRO_FILES:
    path = VIDEO_DIR / name
    if not path.exists():
        print(f'MISSING: {name}')
    else:
        print(f'SHA256: {name} -> {sha256sum(path)}')

NO EXPECTED HASH: GX010063_2025Nov10.MP4 -> 4c8bce5fbf906aa2cd92d6495cf7213f920838024e75a27293becdfaecf7e1c7
NO EXPECTED HASH: GX010067_2025Nov12.MP4 -> 4d38bd222b997e66a7ca2c1633bc1b4b2e2974607aeb9800ed6ea52ee01e2000
NO EXPECTED HASH: GX010072_2025Nov13.MP4 -> dbb8138833d789d0fcde75a88d630b681239815a6d91ee8ac04d70405d1d5675
NO EXPECTED HASH: GX01007X_2025Nov14.MP4 -> f7478c7ea167259af31507490a9546f6e5c5adf89bf4caaf7893300289e769cb
Checksum verification completed: no mismatches detected among files with expected hashes.
